# gold_user_behavior

**Sources:**
- fact A:    `silver.search_events`     (search_id, user_id, query_text, result_count)
- fact B:    `silver.recommendations`   (event_id, user_id, event_type, product_id)
- dimension: `silver.users`             (user_id → cpf, city, country)

**Joins:**
- `search_events.user_id = users.user_id`  (both INT)
- `recommendations.user_id = users.user_id`  (both INT)
- search_events FULL OUTER JOIN recommendations ON user_id, then LEFT JOIN users

**Grain:** one row per user_id  
**Merge key:** `user_id`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog               = dbutils.widgets.get("catalog")
gold_table            = f"{catalog}.gold.user_behavior"
silver_search         = f"{catalog}.silver.search_events"
silver_recommendations = f"{catalog}.silver.recommendations"
silver_users          = f"{catalog}.silver.users"

print(f"[gold] {gold_table}  ←  {silver_search} × {silver_recommendations} × {silver_users}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        user_id                INT NOT NULL,
        cpf                    STRING,
        city                   STRING,
        country                STRING,
        total_searches         LONG,
        avg_results_per_search DOUBLE,
        distinct_queries       LONG,
        total_rec_events       LONG,
        rec_views              LONG,
        rec_clicks             LONG,
        rec_purchases          LONG,
        distinct_products_seen LONG,
        _computed_at           TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (user_id)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import col, count, avg, countDistinct, sum, when, current_timestamp

search_events   = spark.table(silver_search)
recommendations = spark.table(silver_recommendations)
users           = spark.table(silver_users)

# fact A: aggregate search behavior per user_id
search_agg = (
    search_events.groupBy("user_id")
    .agg(
        count("search_id").alias("total_searches"),
        avg("result_count").alias("avg_results_per_search"),
        countDistinct("query_text").alias("distinct_queries"),
    )
)

# fact B: aggregate recommendation events per user_id
# pivot event_type counts using conditional sums
rec_agg = (
    recommendations.groupBy("user_id")
    .agg(
        count("event_id").alias("total_rec_events"),
        sum(when(col("event_type") == "view",     1).otherwise(0)).alias("rec_views"),
        sum(when(col("event_type") == "click",    1).otherwise(0)).alias("rec_clicks"),
        sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("rec_purchases"),
        countDistinct("product_id").alias("distinct_products_seen"),
    )
)

# FULL OUTER JOIN facts so users present in only one source are included
# then LEFT JOIN dimension silver.users for demographics
behavior_df = (
    search_agg
    .join(rec_agg, on="user_id", how="full_outer")
    .join(
        users.select("user_id", "cpf", "city", "country"),
        on="user_id",
        how="left",
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {behavior_df.count()} users with search or recommendation activity")

In [ ]:
behavior_df.createOrReplaceTempView("gold_user_behavior_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_user_behavior_batch AS s
    ON t.user_id = s.user_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")